In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # 多模态脑MRI数据集探索
# 
# 这个notebook用于探索和验证多模态脑MRI数据集的结构完整性。

# ## 1. 导入必要的库

import os
from pathlib import Path
import pandas as pd
from datetime import datetime
import json

# ## 2. 设置数据路径和定义文件结构

# 根目录路径
ROOT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS")

# 定义每个受试者应该包含的文件结构
REQUIRED_FILES = {
    "4D_image": "evaluated/realigned_coregistered/nibabel_stacked_normalized_skull_stripped.nii.gz",
    "3D_label": "seg/converted_alex_labels/resampled_synthseg_t1_mp2rage_alex_labels.nii.gz"
}

# ## 3. 扫描和验证数据集

def scan_dataset(root_dir):
    """
    扫描根目录，找出所有符合条件的受试者文件夹并验证文件完整性
    
    Parameters:
    -----------
    root_dir : Path
        数据集根目录路径
    
    Returns:
    --------
    dict : 包含扫描结果的字典
    """
    
    results = {
        "scan_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "root_directory": str(root_dir),
        "total_subjects": 0,
        "valid_subjects": 0,
        "incomplete_subjects": 0,
        "subjects": []
    }
    
    # 检查根目录是否存在
    if not root_dir.exists():
        print(f"❌ 错误：根目录不存在 - {root_dir}")
        return results
    
    print(f"📁 扫描目录: {root_dir}\n")
    print("=" * 80)
    
    # 查找所有FOR_开头的文件夹
    subject_folders = [f for f in root_dir.iterdir() 
                      if f.is_dir() and f.name.startswith("FOR_")]
    
    results["total_subjects"] = len(subject_folders)
    
    if not subject_folders:
        print("⚠️ 警告：未找到任何以'FOR_'开头的文件夹")
        return results
    
    print(f"🔍 找到 {len(subject_folders)} 个受试者文件夹\n")
    
    # 检查每个受试者文件夹
    for idx, subject_folder in enumerate(subject_folders, 1):
        subject_info = {
            "subject_id": subject_folder.name,
            "path": str(subject_folder),
            "status": "完整",
            "missing_files": [],
            "existing_files": {}
        }
        
        print(f"[{idx}/{len(subject_folders)}] 检查受试者: {subject_folder.name}")
        
        # 检查必需的文件
        all_files_exist = True
        for file_type, relative_path in REQUIRED_FILES.items():
            file_path = subject_folder / relative_path
            
            if file_path.exists():
                subject_info["existing_files"][file_type] = str(file_path)
                print(f"  ✓ {file_type}: 找到")
                
                # 获取文件大小
                file_size_mb = file_path.stat().st_size / (1024 * 1024)
                subject_info["existing_files"][f"{file_type}_size_mb"] = round(file_size_mb, 2)
                
            else:
                all_files_exist = False
                subject_info["missing_files"].append(file_type)
                subject_info["status"] = "不完整"
                print(f"  ✗ {file_type}: 缺失")
        
        if all_files_exist:
            results["valid_subjects"] += 1
            print(f"  状态: ✅ 完整\n")
        else:
            results["incomplete_subjects"] += 1
            print(f"  状态: ⚠️ 不完整 - 缺失 {len(subject_info['missing_files'])} 个文件\n")
        
        results["subjects"].append(subject_info)
    
    return results

# 执行扫描
print("🚀 开始扫描数据集...\n")
scan_results = scan_dataset(ROOT_DIR)

# ## 4. 显示汇总信息

print("\n" + "=" * 80)
print("📊 数据集汇总信息")
print("=" * 80)
print(f"扫描时间: {scan_results['scan_time']}")
print(f"根目录: {scan_results['root_directory']}")
print(f"\n📈 统计信息:")
print(f"  • 总受试者数: {scan_results['total_subjects']}")
print(f"  • 完整数据受试者数: {scan_results['valid_subjects']} ({scan_results['valid_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")
print(f"  • 不完整数据受试者数: {scan_results['incomplete_subjects']} ({scan_results['incomplete_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")

# ## 5. 创建详细的数据框

# 创建受试者信息数据框
subjects_data = []
for subject in scan_results["subjects"]:
    row = {
        "受试者ID": subject["subject_id"],
        "状态": subject["status"],
        "4D影像": "✓" if "4D_image" in subject["existing_files"] else "✗",
        "3D标签": "✓" if "3D_label" in subject["existing_files"] else "✗",
        "缺失文件数": len(subject["missing_files"])
    }
    
    # 添加文件大小信息（如果存在）
    if "4D_image_size_mb" in subject["existing_files"]:
        row["4D影像大小(MB)"] = subject["existing_files"]["4D_image_size_mb"]
    if "3D_label_size_mb" in subject["existing_files"]:
        row["3D标签大小(MB)"] = subject["existing_files"]["3D_label_size_mb"]
    
    subjects_data.append(row)

df_subjects = pd.DataFrame(subjects_data)

print("\n📋 受试者详细信息:")
print(df_subjects.to_string(index=False))

# ## 6. 提取有效受试者路径列表

valid_subject_paths = []
valid_subject_info = []

for subject in scan_results["subjects"]:
    if subject["status"] == "完整":
        valid_subject_paths.append(subject["path"])
        valid_subject_info.append({
            "subject_id": subject["subject_id"],
            "path": subject["path"],
            "4d_image_path": subject["existing_files"]["4D_image"],
            "3d_label_path": subject["existing_files"]["3D_label"]
        })

print(f"\n✅ 找到 {len(valid_subject_paths)} 个数据完整的受试者")
print("\n有效受试者路径列表:")
for i, path in enumerate(valid_subject_paths, 1):
    print(f"{i}. {path}")

# ## 7. 保存结果

# 保存扫描结果为JSON文件
output_dir = Path("./mri_dataset_analysis_results")
output_dir.mkdir(exist_ok=True)

# 保存完整扫描结果
scan_results_file = output_dir / f"scan_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(scan_results_file, 'w', encoding='utf-8') as f:
    json.dump(scan_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 完整扫描结果已保存到: {scan_results_file}")

# 保存有效受试者信息
valid_subjects_file = output_dir / "valid_subjects.json"
with open(valid_subjects_file, 'w', encoding='utf-8') as f:
    json.dump(valid_subject_info, f, ensure_ascii=False, indent=2)

print(f"💾 有效受试者信息已保存到: {valid_subjects_file}")

# 保存为CSV格式（便于在Excel中查看）
csv_file = output_dir / f"subjects_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_subjects.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"💾 受试者汇总表已保存到: {csv_file}")

# ## 8. 数据质量检查

print("\n" + "=" * 80)
print("🔍 数据质量检查")
print("=" * 80)

# 检查不完整的受试者
incomplete_subjects = [s for s in scan_results["subjects"] if s["status"] == "不完整"]

if incomplete_subjects:
    print(f"\n⚠️ 发现 {len(incomplete_subjects)} 个数据不完整的受试者:")
    for subject in incomplete_subjects:
        print(f"\n受试者: {subject['subject_id']}")
        print(f"缺失文件: {', '.join(subject['missing_files'])}")
else:
    print("\n✅ 所有受试者数据完整！")

# 文件大小统计（仅针对完整数据）
if valid_subject_info:
    print("\n📊 文件大小统计（仅完整数据）:")
    
    sizes_4d = [s["existing_files"].get("4D_image_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    sizes_3d = [s["existing_files"].get("3D_label_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    
    if sizes_4d:
        print(f"\n4D影像文件:")
        print(f"  • 平均大小: {sum(sizes_4d)/len(sizes_4d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_4d):.2f} / {max(sizes_4d):.2f} MB")
    
    if sizes_3d:
        print(f"\n3D标签文件:")
        print(f"  • 平均大小: {sum(sizes_3d)/len(sizes_3d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_3d):.2f} / {max(sizes_3d):.2f} MB")

# ## 9. 快速访问有效数据的辅助函数

def get_subject_files(subject_id, valid_subjects=valid_subject_info):
    """
    根据受试者ID获取其文件路径
    
    Parameters:
    -----------
    subject_id : str
        受试者ID
    valid_subjects : list
        有效受试者信息列表
    
    Returns:
    --------
    dict : 包含文件路径的字典，如果未找到则返回None
    """
    for subject in valid_subjects:
        if subject["subject_id"] == subject_id:
            return {
                "4d_image": Path(subject["4d_image_path"]),
                "3d_label": Path(subject["3d_label_path"])
            }
    return None

# 示例：如何使用这个函数
if valid_subject_info:
    example_subject = valid_subject_info[0]["subject_id"]
    files = get_subject_files(example_subject)
    print(f"\n📌 示例：获取受试者 '{example_subject}' 的文件路径:")
    if files:
        print(f"  • 4D影像: {files['4d_image']}")
        print(f"  • 3D标签: {files['3d_label']}")

print("\n✨ 数据集探索完成！")
print(f"📁 所有结果已保存到: {output_dir.absolute()}")

# 将有效受试者路径列表存储为变量，便于后续使用
print(f"\n💡 提示：变量 'valid_subject_paths' 包含了所有数据完整的受试者路径")
print(f"        变量 'valid_subject_info' 包含了详细的文件路径信息")

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # NIfTI数据检查与分析
# 
# 这个notebook用于读取和检查多模态脑MRI数据的NIfTI文件

# ## 1. 导入必要的库

import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# 设置显示选项
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ## 2. 加载之前保存的有效受试者信息

# 加载有效受试者信息
valid_subjects_file = Path("./mri_dataset_analysis_results/valid_subjects.json")

if not valid_subjects_file.exists():
    print("❌ 错误：找不到valid_subjects.json文件")
    print("请先运行数据集探索notebook")
else:
    with open(valid_subjects_file, 'r', encoding='utf-8') as f:
        valid_subject_info = json.load(f)
    
    print(f"✅ 成功加载 {len(valid_subject_info)} 个有效受试者信息")

# ## 3. 读取第一个受试者的数据

if valid_subject_info:
    # 获取第一个受试者
    first_subject = valid_subject_info[0]
    print(f"\n📊 分析受试者: {first_subject['subject_id']}")
    print("=" * 80)
    
    # 读取4D影像和3D标签
    print("\n🔄 加载NIfTI文件...")
    
    # 4D影像
    img_4d_path = Path(first_subject['4d_image_path'])
    img_4d = nib.load(img_4d_path)
    data_4d = img_4d.get_fdata()
    
    print(f"✓ 4D影像加载完成: {img_4d_path.name}")
    
    # 3D标签
    label_3d_path = Path(first_subject['3d_label_path'])
    label_3d = nib.load(label_3d_path)
    data_label = label_3d.get_fdata()
    
    print(f"✓ 3D标签加载完成: {label_3d_path.name}")

# ## 4. 检查数据形状

print("\n📐 数据形状检查:")
print("=" * 80)

# 4D影像形状
expected_4d_shape = (288, 336, 384, 42)
actual_4d_shape = data_4d.shape
shape_match_4d = actual_4d_shape == expected_4d_shape

print(f"\n4D影像:")
print(f"  • 期望形状: {expected_4d_shape}")
print(f"  • 实际形状: {actual_4d_shape}")
print(f"  • 匹配状态: {'✅ 匹配' if shape_match_4d else '❌ 不匹配'}")

# 3D标签形状
expected_3d_shape = (288, 336, 384)
actual_3d_shape = data_label.shape
shape_match_3d = actual_3d_shape == expected_3d_shape

print(f"\n3D标签:")
print(f"  • 期望形状: {expected_3d_shape}")
print(f"  • 实际形状: {actual_3d_shape}")
print(f"  • 匹配状态: {'✅ 匹配' if shape_match_3d else '❌ 不匹配'}")

# ## 5. 检查头信息

print("\n🔍 NIfTI头信息:")
print("=" * 80)

# 4D影像头信息
print("\n📊 4D影像头信息:")
print(f"  • 数据类型: {img_4d.header.get_data_dtype()}")
print(f"  • 体素尺寸: {img_4d.header.get_zooms()[:3]} mm")
print(f"  • TR (如果可用): {img_4d.header.get_zooms()[3] if len(img_4d.header.get_zooms()) > 3 else 'N/A'} s")
print(f"  • 数据方向: {nib.aff2axcodes(img_4d.affine)}")

print("\n  • 仿射矩阵:")
print(img_4d.affine)

# 3D标签头信息
print("\n📊 3D标签头信息:")
print(f"  • 数据类型: {label_3d.header.get_data_dtype()}")
print(f"  • 体素尺寸: {label_3d.header.get_zooms()} mm")
print(f"  • 数据方向: {nib.aff2axcodes(label_3d.affine)}")

print("\n  • 仿射矩阵:")
print(label_3d.affine)

# 检查仿射矩阵是否匹配
affine_match = np.allclose(img_4d.affine, label_3d.affine, rtol=1e-5)
print(f"\n⚡ 仿射矩阵匹配: {'✅ 是' if affine_match else '❌ 否'}")

# ## 6. 4D影像数据统计分析

print("\n📊 4D影像数据统计:")
print("=" * 80)

# 计算每个时间点的统计信息
stats_per_timepoint = []
for t in range(data_4d.shape[3]):
    volume = data_4d[:, :, :, t]
    stats = {
        '时间点': t,
        '最小值': np.min(volume),
        '最大值': np.max(volume),
        '均值': np.mean(volume),
        '标准差': np.std(volume),
        '中位数': np.median(volume)
    }
    stats_per_timepoint.append(stats)

# 创建统计数据框
df_stats = pd.DataFrame(stats_per_timepoint)

print("\n各时间点统计信息摘要:")
print(df_stats.describe())

# 检查是否已经进行了Z-score标准化
print("\n🔍 检查数据标准化状态:")
global_mean = np.mean(data_4d)
global_std = np.std(data_4d)
print(f"  • 全局均值: {global_mean:.6f}")
print(f"  • 全局标准差: {global_std:.6f}")

# 判断是否接近标准正态分布
is_z_scored = abs(global_mean) < 0.1 and abs(global_std - 1.0) < 0.1
print(f"  • Z-score标准化: {'✅ 可能已标准化' if is_z_scored else '❌ 未标准化'}")

# 检查每个通道的分布
print("\n📈 前5个时间点的详细统计:")
print(df_stats.head())

# ## 7. 3D标签数据分析

print("\n🏷️ 3D标签数据分析:")
print("=" * 80)

# 获取唯一标签值
unique_labels = np.unique(data_label)
print(f"\n发现 {len(unique_labels)} 个唯一标签值")

# 统计每个标签的体素数量
label_counts = []
for label in unique_labels:
    count = np.sum(data_label == label)
    percentage = (count / data_label.size) * 100
    label_counts.append({
        '标签值': int(label),
        '体素数量': count,
        '占比(%)': percentage
    })

# 创建标签统计数据框
df_labels = pd.DataFrame(label_counts)
df_labels = df_labels.sort_values('标签值')

print("\n标签分布统计:")
print(df_labels.to_string(index=False))

# 计算总体素数
total_voxels = data_label.size
print(f"\n总体素数: {total_voxels:,}")
print(f"验证总和: {df_labels['体素数量'].sum():,} ({'✅ 匹配' if df_labels['体素数量'].sum() == total_voxels else '❌ 不匹配'})")

# ## 8. 检查标签的合理性

print("\n🔍 标签合理性检查:")
print("=" * 80)

# 定义常见的FreeSurfer/SynthSeg标签范围
# 这些是典型的脑区标签值，你可以根据实际情况调整
common_brain_labels = {
    0: "背景/CSF",
    2: "左侧大脑白质",
    3: "左侧大脑皮质",
    4: "左侧侧脑室",
    7: "左侧小脑白质", 
    8: "左侧小脑皮质",
    10: "左侧丘脑",
    11: "左侧尾状核",
    12: "左侧壳核",
    13: "左侧苍白球",
    14: "第三脑室",
    15: "第四脑室",
    16: "脑干",
    17: "左侧海马",
    18: "左侧杏仁核",
    24: "脑脊液",
    26: "左侧伏隔核",
    28: "左侧腹侧间脑",
    41: "右侧大脑白质",
    42: "右侧大脑皮质",
    43: "右侧侧脑室",
    46: "右侧小脑白质",
    47: "右侧小脑皮质",
    49: "右侧丘脑",
    50: "右侧尾状核",
    51: "右侧壳核",
    52: "右侧苍白球",
    53: "右侧海马",
    54: "右侧杏仁核",
    58: "右侧伏隔核",
    60: "右侧腹侧间脑"
}

# 检查是否有意外的标签值
expected_labels = set(common_brain_labels.keys())
actual_labels = set(int(label) for label in unique_labels)

# 找出意外的标签
unexpected_labels = actual_labels - expected_labels
missing_labels = expected_labels - actual_labels

if unexpected_labels:
    print(f"\n⚠️ 发现意外的标签值: {sorted(unexpected_labels)}")
else:
    print("\n✅ 所有标签值都在预期范围内")

# 显示标签含义（如果已知）
print("\n📋 标签含义对照表:")
for label in sorted(actual_labels):
    if label in common_brain_labels:
        voxel_count = df_labels[df_labels['标签值'] == label]['体素数量'].values[0]
        percentage = df_labels[df_labels['标签值'] == label]['占比(%)'].values[0]
        print(f"  {label:3d}: {common_brain_labels[label]:20s} - {voxel_count:8,d} 体素 ({percentage:5.2f}%)")
    else:
        voxel_count = df_labels[df_labels['标签值'] == label]['体素数量'].values[0]
        percentage = df_labels[df_labels['标签值'] == label]['占比(%)'].values[0]
        print(f"  {label:3d}: {'未知区域':20s} - {voxel_count:8,d} 体素 ({percentage:5.2f}%)")

# ## 9. 可视化标签分布

# 创建标签分布的柱状图
plt.figure(figsize=(12, 6))
plt.bar(df_labels['标签值'].astype(str), df_labels['占比(%)'])
plt.xlabel('标签值')
plt.ylabel('占比 (%)')
plt.title(f'标签分布 - {first_subject["subject_id"]}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ## 10. 数据质量总结

print("\n" + "=" * 80)
print("📋 数据质量总结")
print("=" * 80)

quality_checks = [
    ("4D影像形状", shape_match_4d),
    ("3D标签形状", shape_match_3d),
    ("仿射矩阵匹配", affine_match),
    ("标签值合理性", len(unexpected_labels) == 0),
    ("数据完整性", df_labels['体素数量'].sum() == total_voxels)
]

all_passed = True
for check_name, passed in quality_checks:
    status = "✅ 通过" if passed else "❌ 失败"
    print(f"{check_name}: {status}")
    all_passed = all_passed and passed

print("\n" + "=" * 80)
if all_passed:
    print("✨ 总体评估: 数据质量良好，所有检查项都通过！")
else:
    print("⚠️ 总体评估: 发现一些问题，请检查上述失败项。")

# ## 11. 保存检查结果

# 准备检查结果字典
check_results = {
    "subject_id": first_subject["subject_id"],
    "check_time": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "4d_shape": list(actual_4d_shape),
    "3d_shape": list(actual_3d_shape),
    "shape_match": {
        "4d": shape_match_4d,
        "3d": shape_match_3d
    },
    "data_stats": {
        "global_mean": float(global_mean),
        "global_std": float(global_std),
        "is_z_scored": is_z_scored
    },
    "label_info": {
        "unique_labels": [int(l) for l in unique_labels],
        "label_counts": df_labels.to_dict('records'),
        "unexpected_labels": list(unexpected_labels)
    },
    "quality_passed": all_passed
}

# 保存结果
output_file = Path("./mri_dataset_analysis_results/data_check_results.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(check_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 检查结果已保存到: {output_file}")

# ## 12. 批量检查所有受试者（可选）

print("\n" + "=" * 80)
print("🔄 批量检查所有受试者")
print("=" * 80)

batch_check = input("\n是否要检查所有受试者的数据一致性？(y/n): ")

if batch_check.lower() == 'y':
    all_subjects_results = []
    
    for subject in valid_subject_info:
        print(f"\n检查 {subject['subject_id']}...", end='')
        
        try:
            # 加载数据
            img = nib.load(subject['4d_image_path'])
            label = nib.load(subject['3d_label_path'])
            
            # 获取形状
            img_shape = img.shape
            label_shape = label.shape
            
            # 检查形状
            shape_ok = (img_shape == expected_4d_shape and 
                       label_shape == expected_3d_shape)
            
            result = {
                'subject_id': subject['subject_id'],
                '4d_shape': img_shape,
                '3d_shape': label_shape,
                'shape_ok': shape_ok
            }
            
            all_subjects_results.append(result)
            print(" ✓" if shape_ok else " ✗")
            
        except Exception as e:
            print(f" ❌ 错误: {str(e)}")
            all_subjects_results.append({
                'subject_id': subject['subject_id'],
                'error': str(e)
            })
    
    # 汇总结果
    df_batch = pd.DataFrame(all_subjects_results)
    
    print("\n批量检查结果汇总:")
    print(df_batch)
    
    # 保存批量检查结果
    batch_file = Path("./mri_dataset_analysis_results/batch_check_results.csv")
    df_batch.to_csv(batch_file, index=False)
    print(f"\n💾 批量检查结果已保存到: {batch_file}")

print("\n✅ 数据检查完成！")

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # MRI数据展平处理 - 适配Fortran顺序
# 
# 将3D/4D MRI数据展平为1D/2D格式，用于后续训练

# ## 1. 导入必要的库

import nibabel as nib
import numpy as np
import json
from pathlib import Path
import time
from datetime import datetime
import os

# ## 2. 定义数据展平和还原函数

def flatten_mri_data(data_4d, label_3d, order='F'):
    """
    将4D影像和3D标签展平
    
    Parameters:
    -----------
    data_4d : np.ndarray
        形状为 (x, y, z, n_modalities) 的4D数组
    label_3d : np.ndarray
        形状为 (x, y, z) 的3D标签数组
    order : str
        展平顺序，'F' for Fortran (列优先) 或 'C' for C (行优先)
        默认使用'F'以匹配你的函数
    
    Returns:
    --------
    features : np.ndarray
        形状为 (n_voxels, n_modalities) 的2D数组
    labels : np.ndarray
        形状为 (n_voxels,) 的1D数组
    """
    # 获取维度信息
    x_dim, y_dim, z_dim, n_modalities = data_4d.shape
    n_voxels = x_dim * y_dim * z_dim
    
    # 重塑4D数据为2D
    # 先将前3个维度展平，保持最后一个维度
    features = data_4d.reshape(-1, n_modalities, order=order)
    
    # 展平3D标签为1D
    labels = label_3d.flatten(order=order)
    
    return features, labels

def create_region_mask(label_3d, background_label=0):
    """
    创建区域掩码（非背景体素）
    
    Parameters:
    -----------
    label_3d : np.ndarray
        3D标签数组
    background_label : int
        背景标签值，默认为0
    
    Returns:
    --------
    region : np.ndarray
        3D布尔数组，True表示非背景体素
    """
    region = label_3d != background_label
    return region

def verify_flattening(data_4d, label_3d, features, labels, order='F'):
    """
    验证展平的正确性
    
    Parameters:
    -----------
    data_4d, label_3d : 原始数据
    features, labels : 展平后的数据
    order : 展平顺序
    
    Returns:
    --------
    bool : 验证是否通过
    """
    # 获取原始形状
    x_dim, y_dim, z_dim, n_modalities = data_4d.shape
    
    # 随机检查一些点
    n_checks = 100
    passed = True
    
    print(f"\n验证展平正确性（检查{n_checks}个随机点）...")
    
    for _ in range(n_checks):
        # 随机选择一个3D坐标
        x = np.random.randint(0, x_dim)
        y = np.random.randint(0, y_dim)
        z = np.random.randint(0, z_dim)
        
        # 计算展平后的索引（使用相同的顺序）
        if order == 'F':  # Fortran order
            flat_idx = x + y * x_dim + z * x_dim * y_dim
        else:  # C order
            flat_idx = z + y * z_dim + x * z_dim * y_dim
        
        # 检查数据是否匹配
        original_features = data_4d[x, y, z, :]
        flattened_features = features[flat_idx, :]
        
        original_label = label_3d[x, y, z]
        flattened_label = labels[flat_idx]
        
        if not np.allclose(original_features, flattened_features) or original_label != flattened_label:
            print(f"❌ 不匹配: 位置({x},{y},{z}) -> 索引{flat_idx}")
            passed = False
            break
    
    if passed:
        print("✅ 所有检查点都匹配！")
    
    return passed

def get_spectra_at_voxel(features, x, y, z, shape, order='F'):
    """
    获取指定体素的所有模态数据（用于验证）
    
    Parameters:
    -----------
    features : 2D array (n_voxels, n_modalities)
    x, y, z : 体素坐标
    shape : 原始3D形状 (x_dim, y_dim, z_dim)
    order : 展平顺序
    
    Returns:
    --------
    spectra : 1D array (n_modalities,)
    """
    x_dim, y_dim, z_dim = shape
    
    if order == 'F':
        flat_idx = x + y * x_dim + z * x_dim * y_dim
    else:
        flat_idx = z + y * z_dim + x * z_dim * y_dim
    
    return features[flat_idx, :]

def revert_reshape_simple(array, shape, order='F'):
    """
    简单版本的还原函数，将1D/2D数组还原为3D/4D
    
    Parameters:
    -----------
    array : 1D or 2D array
    shape : 目标3D形状
    order : 展平顺序
    
    Returns:
    --------
    reshaped : 3D or 4D array
    """
    if array.ndim == 1:
        # 1D -> 3D
        return array.reshape(shape, order=order)
    else:
        # 2D -> 4D
        n_features = array.shape[1]
        return array.reshape((*shape, n_features), order=order)

# ## 3. 主处理函数

def process_subject(subject_info, verify=True):
    """
    处理单个受试者的数据，将结果保存到原始数据目录下
    
    Parameters:
    -----------
    subject_info : dict
        包含受试者信息的字典
    verify : bool
        是否验证展平正确性
    
    Returns:
    --------
    dict : 处理结果信息
    """
    subject_id = subject_info['subject_id']
    print(f"\n{'='*80}")
    print(f"处理受试者: {subject_id}")
    print(f"{'='*80}")
    
    # 使用原始数据路径，创建flattened子目录
    subject_path = Path(subject_info['path'])
    subject_output_dir = subject_path / 'flattened'
    subject_output_dir.mkdir(parents=True, exist_ok=True)
    
    # 加载数据
    print("📂 加载NIfTI文件...")
    start_time = time.time()
    
    img_4d = nib.load(subject_info['4d_image_path'])
    data_4d = img_4d.get_fdata()
    
    label_3d = nib.load(subject_info['3d_label_path'])
    data_label = label_3d.get_fdata().astype(np.int32)
    
    load_time = time.time() - start_time
    print(f"✓ 加载完成 (耗时: {load_time:.2f}秒)")
    
    # 获取数据信息
    shape_3d = data_label.shape
    n_modalities = data_4d.shape[3]
    n_voxels = np.prod(shape_3d)
    
    print(f"\n📊 数据信息:")
    print(f"  • 3D形状: {shape_3d}")
    print(f"  • 模态数: {n_modalities}")
    print(f"  • 总体素数: {n_voxels:,}")
    
    # 展平数据
    print("\n🔄 展平数据...")
    start_time = time.time()
    
    features, labels = flatten_mri_data(data_4d, data_label, order='F')
    
    flatten_time = time.time() - start_time
    print(f"✓ 展平完成 (耗时: {flatten_time:.2f}秒)")
    print(f"  • Features形状: {features.shape}")
    print(f"  • Labels形状: {labels.shape}")
    
    # 创建区域掩码
    region = create_region_mask(data_label)
    n_valid_voxels = np.sum(region)
    print(f"  • 非背景体素数: {n_valid_voxels:,} ({n_valid_voxels/n_voxels*100:.2f}%)")
    
    # 验证展平正确性
    if verify:
        verify_result = verify_flattening(data_4d, data_label, features, labels, order='F')
        if not verify_result:
            print("❌ 验证失败！")
            return None
    
    # 转换数据类型以节省空间
    features = features.astype(np.float32)
    labels = labels.astype(np.int32)
    
    # 保存数据
    print("\n💾 保存数据...")
    start_time = time.time()
    
    # 保存展平的数据
    np.save(subject_output_dir / 'features.npy', features)
    np.save(subject_output_dir / 'labels.npy', labels)
    np.save(subject_output_dir / 'region_mask.npy', region)
    
    # 保存元数据
    metadata = {
        "subject_id": subject_id,
        "processing_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "original_shape_3d": list(shape_3d),
        "n_modalities": n_modalities,
        "n_voxels_total": int(n_voxels),
        "n_voxels_valid": int(n_valid_voxels),
        "flatten_order": "F",  # Fortran order
        "data_types": {
            "features": "float32",
            "labels": "int32"
        },
        "file_sizes_mb": {
            "features": os.path.getsize(subject_output_dir / 'features.npy') / (1024**2),
            "labels": os.path.getsize(subject_output_dir / 'labels.npy') / (1024**2),
            "region_mask": os.path.getsize(subject_output_dir / 'region_mask.npy') / (1024**2)
        },
        "label_statistics": {
            "unique_labels": sorted([int(l) for l in np.unique(labels)]),
            "n_unique_labels": len(np.unique(labels))
        }
    }
    
    with open(subject_output_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    save_time = time.time() - start_time
    print(f"✓ 保存完成 (耗时: {save_time:.2f}秒)")
    print(f"  • 输出目录: {subject_output_dir}")
    
    # 返回处理结果
    result = {
        "subject_id": subject_id,
        "success": True,
        "output_dir": str(subject_output_dir),
        "processing_times": {
            "load": load_time,
            "flatten": flatten_time,
            "save": save_time,
            "total": load_time + flatten_time + save_time
        },
        "data_info": metadata
    }
    
    return result

# ## 4. 验证数据可以正确还原

def verify_data_recovery(subject_output_dir):
    """
    验证保存的数据可以正确还原
    """
    print(f"\n🔍 验证数据还原: {subject_output_dir}")
    
    # 加载保存的数据
    features = np.load(subject_output_dir / 'features.npy')
    labels = np.load(subject_output_dir / 'labels.npy')
    region = np.load(subject_output_dir / 'region_mask.npy')
    
    with open(subject_output_dir / 'metadata.json', 'r') as f:
        metadata = json.load(f)
    
    shape_3d = tuple(metadata['original_shape_3d'])
    n_modalities = metadata['n_modalities']
    
    print(f"  • 加载的features形状: {features.shape}")
    print(f"  • 加载的labels形状: {labels.shape}")
    print(f"  • 原始3D形状: {shape_3d}")
    
    # 还原为3D/4D
    labels_3d = revert_reshape_simple(labels, shape_3d, order='F')
    features_4d = revert_reshape_simple(features, shape_3d, order='F')
    
    print(f"  • 还原后labels形状: {labels_3d.shape}")
    print(f"  • 还原后features形状: {features_4d.shape}")
    
    # 测试get_spectra_at_slice风格的函数
    print("\n  测试切片提取...")
    z = shape_3d[2] // 2  # 中间切片
    
    # 提取切片数据
    slice_data = get_slice_spectra(features, region, z, shape_3d)
    print(f"  • 切片{z}的数据形状: {slice_data.shape}")
    
    # 验证切片数据
    # 直接从4D数据提取相同切片进行比较
    direct_slice = features_4d[:, :, z, :]
    
    # 考虑掩码的影响
    match = np.allclose(slice_data[region[:, :, z]], direct_slice[region[:, :, z]])
    print(f"  • 切片数据匹配: {'✅' if match else '❌'}")
    
    return match

def get_slice_spectra(features, region, z, shape_3d):
    """
    获取指定切片的所有模态数据（适配你的函数风格）
    """
    x_dim, y_dim, z_dim = shape_3d
    n_modalities = features.shape[1]
    
    # 初始化切片数据
    slice_spectra = np.zeros((x_dim, y_dim, n_modalities), dtype=features.dtype)
    
    # 获取切片的掩码
    region_slice = region[:, :, z]
    
    # 计算每个位置的展平索引
    x_indices, y_indices = np.meshgrid(np.arange(x_dim), np.arange(y_dim), indexing='ij')
    
    # 对于掩码中的每个有效位置
    valid_positions = np.where(region_slice)
    
    for i, (x, y) in enumerate(zip(valid_positions[0], valid_positions[1])):
        # 计算展平索引（Fortran order）
        flat_idx = x + y * x_dim + z * x_dim * y_dim
        slice_spectra[x, y, :] = features[flat_idx, :]
    
    return slice_spectra

# ## 5. 批量处理所有受试者

def process_all_subjects(valid_subjects_file, max_subjects=None):
    """
    批量处理所有受试者，将数据保存到各自的原始目录下
    """
    # 加载受试者信息
    with open(valid_subjects_file, 'r') as f:
        valid_subjects = json.load(f)
    
    if max_subjects:
        valid_subjects = valid_subjects[:max_subjects]
    
    n_subjects = len(valid_subjects)
    print(f"\n🚀 开始批量处理 {n_subjects} 个受试者")
    print(f"数据将保存到各受试者原始目录的 'flattened' 子文件夹中")
    
    # 处理每个受试者
    results = []
    total_start_time = time.time()
    
    for i, subject_info in enumerate(valid_subjects):
        print(f"\n[{i+1}/{n_subjects}] ", end='')
        
        try:
            result = process_subject(subject_info, verify=True)
            results.append(result)
            
            # 验证数据可以正确还原
            if result and result['success']:
                verify_data_recovery(Path(result['output_dir']))
                
        except Exception as e:
            print(f"❌ 处理失败: {str(e)}")
            results.append({
                "subject_id": subject_info['subject_id'],
                "success": False,
                "error": str(e)
            })
    
    total_time = time.time() - total_start_time
    
    # 汇总结果
    print(f"\n{'='*80}")
    print("📊 批量处理完成")
    print(f"{'='*80}")
    
    successful = sum(1 for r in results if r['success'])
    print(f"成功: {successful}/{n_subjects}")
    print(f"总耗时: {total_time:.2f}秒 (平均: {total_time/n_subjects:.2f}秒/受试者)")
    
    # 保存处理汇总到第一个受试者的父目录
    if valid_subjects:
        parent_dir = Path(valid_subjects[0]['path']).parent
        summary_file = parent_dir / 'flattening_summary.json'
    else:
        summary_file = Path('./flattening_summary.json')
    
    summary = {
        "processing_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_subjects": n_subjects,
        "successful": successful,
        "total_time_seconds": total_time,
        "results": results
    }
    
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n💾 处理汇总已保存到: {summary_file}")
    
    return results

# ## 6. 主程序

if __name__ == "__main__":
    # 设置路径
    valid_subjects_file = Path("./mri_dataset_analysis_results/valid_subjects.json")
    
    # 处理所有受试者
    # 可以设置max_subjects=1来只处理第一个受试者进行测试
    results = process_all_subjects(valid_subjects_file, max_subjects=1)
    
    print("\n✅ 所有处理完成！")
    
    # 显示数据保存位置
    if results and results[0]['success']:
        example_path = Path(results[0]['output_dir'])
        print(f"\n📁 数据保存位置示例:")
        print(f"   {example_path}")
        print(f"   包含: features.npy, labels.npy, region_mask.npy, metadata.json")
    
    # 提示如何使用数据
    print("\n💡 使用展平数据的示例代码:")
    print("""
# 加载一个受试者的数据
import numpy as np
from pathlib import Path

# 使用受试者原始路径
subject_path = Path('/path/to/FOR_016_20250204_reproducibility')
flattened_dir = subject_path / 'flattened'

features = np.load(flattened_dir / 'features.npy')  # (37158912, 42)
labels = np.load(flattened_dir / 'labels.npy')      # (37158912,)
region = np.load(flattened_dir / 'region_mask.npy') # (288, 336, 384)

# 获取非背景体素
valid_voxels = region.flatten(order='F')
features_valid = features[valid_voxels]
labels_valid = labels[valid_voxels]

print(f"非背景体素数: {features_valid.shape[0]:,}")
""")

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # 标签映射工具 - 将稀疏标签映射到连续值
# 
# 将原始的52个稀疏标签值映射到连续的0-51范围

import numpy as np
import json
from pathlib import Path
import time
from datetime import datetime

# ## 1. 创建标签映射

def create_label_mappings(unique_labels):
    """
    创建双向标签映射字典
    
    Parameters:
    -----------
    unique_labels : array-like
        原始数据中的唯一标签值（已排序）
    
    Returns:
    --------
    forward_mapping : dict
        原始值 → 连续值的映射字典
    reverse_mapping : dict
        连续值 → 原始值的映射字典
    """
    # 确保标签是排序的
    unique_labels = sorted([int(label) for label in unique_labels])
    
    # 创建映射
    forward_mapping = {original: continuous for continuous, original in enumerate(unique_labels)}
    reverse_mapping = {continuous: original for continuous, original in enumerate(unique_labels)}
    
    return forward_mapping, reverse_mapping

# 定义标准的52个标签值（基于你的数据）
STANDARD_LABELS = [
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17,
    29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
    44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
    59, 60, 61, 62, 103
]

# 创建标准映射
FORWARD_MAPPING, REVERSE_MAPPING = create_label_mappings(STANDARD_LABELS)

print("标签映射示例：")
print(f"原始标签 → 连续标签")
for i, (orig, cont) in enumerate(FORWARD_MAPPING.items()):
    if i < 10 or i >= len(FORWARD_MAPPING) - 5:
        print(f"  {orig:3d} → {cont:2d}")
    elif i == 10:
        print("  ...")

# ## 2. 标签映射函数

def map_labels(labels, mapping_dict, label_name="labels"):
    """
    安全地映射标签，包含错误检查
    
    Parameters:
    -----------
    labels : np.ndarray
        原始标签数组
    mapping_dict : dict
        映射字典（原始值 → 目标值）
    label_name : str
        标签名称，用于错误消息
    
    Returns:
    --------
    mapped_labels : np.ndarray
        映射后的标签数组
    
    Raises:
    -------
    ValueError : 如果发现未知的标签值
    """
    print(f"\n🔄 映射{label_name}...")
    
    # 获取唯一值
    unique_values = np.unique(labels)
    print(f"  • 发现 {len(unique_values)} 个唯一标签值")
    
    # 检查是否有未知标签
    unknown_labels = []
    for label in unique_values:
        if int(label) not in mapping_dict:
            unknown_labels.append(int(label))
    
    if unknown_labels:
        raise ValueError(
            f"发现未知的标签值: {unknown_labels}\n"
            f"期望的标签值: {sorted(mapping_dict.keys())}"
        )
    
    # 创建映射后的数组
    mapped_labels = np.zeros_like(labels)
    
    # 应用映射
    for original, continuous in mapping_dict.items():
        mask = labels == original
        mapped_labels[mask] = continuous
        count = np.sum(mask)
        if count > 0:
            print(f"  • {original:3d} → {continuous:2d} ({count:,} 个体素)")
    
    # 验证映射
    mapped_unique = np.unique(mapped_labels)
    expected_continuous = sorted(set(mapping_dict.values()))
    
    print(f"\n  ✓ 映射完成")
    print(f"  • 映射后的标签范围: {mapped_unique.min()} - {mapped_unique.max()}")
    print(f"  • 映射后的唯一值数量: {len(mapped_unique)}")
    
    return mapped_labels

# ## 3. One-hot编码函数

def labels_to_onehot(labels, num_classes=52):
    """
    将标签转换为one-hot编码
    
    Parameters:
    -----------
    labels : np.ndarray
        标签数组（应该是连续的0到num_classes-1）
    num_classes : int
        类别数量，默认52
    
    Returns:
    --------
    onehot : np.ndarray
        One-hot编码数组，形状为 (n_samples, num_classes)
    """
    # 检查标签范围
    min_label = labels.min()
    max_label = labels.max()
    
    if min_label < 0:
        raise ValueError(f"标签包含负值: {min_label}")
    if max_label >= num_classes:
        raise ValueError(f"标签值 {max_label} 超出范围 [0, {num_classes-1}]")
    
    # 创建one-hot编码
    n_samples = labels.shape[0]
    onehot = np.zeros((n_samples, num_classes), dtype=np.float32)
    onehot[np.arange(n_samples), labels] = 1
    
    return onehot

# ## 4. 处理单个受试者的标签映射

def process_subject_labels(subject_path, forward_mapping=None, save_onehot=False):
    """
    处理单个受试者的标签映射
    
    Parameters:
    -----------
    subject_path : str or Path
        受试者的flattened数据目录路径
    forward_mapping : dict or None
        标签映射字典，如果为None则使用标准映射
    save_onehot : bool
        是否保存one-hot编码版本
    
    Returns:
    --------
    dict : 处理结果信息
    """
    subject_path = Path(subject_path)
    
    if forward_mapping is None:
        forward_mapping = FORWARD_MAPPING
    
    print(f"\n{'='*80}")
    print(f"处理受试者: {subject_path.parent.name}")
    print(f"{'='*80}")
    
    try:
        # 加载原始标签
        labels_path = subject_path / 'labels.npy'
        if not labels_path.exists():
            raise FileNotFoundError(f"找不到标签文件: {labels_path}")
        
        labels = np.load(labels_path)
        print(f"✓ 加载标签: {labels.shape}")
        
        # 应用标签映射
        labels_mapped = map_labels(labels, forward_mapping)
        
        # 保存映射后的标签
        mapped_path = subject_path / 'labels_mapped.npy'
        np.save(mapped_path, labels_mapped.astype(np.int32))
        print(f"\n💾 保存映射后的标签: {mapped_path.name}")
        
        # 如果需要，保存one-hot编码
        if save_onehot:
            print("\n🔄 创建one-hot编码...")
            # 只对非背景体素创建one-hot（节省空间）
            region = np.load(subject_path / 'region_mask.npy')
            valid_mask = region.flatten(order='F')
            
            labels_valid = labels_mapped[valid_mask]
            onehot_valid = labels_to_onehot(labels_valid, num_classes=52)
            
            onehot_path = subject_path / 'labels_onehot_valid.npy'
            np.save(onehot_path, onehot_valid)
            print(f"✓ 保存one-hot编码（仅非背景）: {onehot_valid.shape}")
        
        # 更新metadata
        metadata_path = subject_path / 'metadata.json'
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        # 添加映射信息
        metadata['label_mapping'] = {
            'mapping_applied': True,
            'mapping_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'num_classes': 52,
            'forward_mapping': {str(k): v for k, v in forward_mapping.items()},
            'reverse_mapping': {str(k): v for k, v in REVERSE_MAPPING.items()},
            'original_unique_labels': sorted([int(k) for k in forward_mapping.keys()]),
            'continuous_labels': list(range(52))
        }
        
        # 更新标签统计
        unique_mapped = np.unique(labels_mapped)
        metadata['mapped_label_statistics'] = {
            'unique_labels': [int(l) for l in unique_mapped],
            'n_unique_labels': len(unique_mapped),
            'min_label': int(unique_mapped.min()),
            'max_label': int(unique_mapped.max())
        }
        
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print("\n✅ 标签映射完成！")
        
        return {
            'success': True,
            'subject': subject_path.parent.name,
            'labels_shape': labels.shape,
            'unique_original': len(np.unique(labels)),
            'unique_mapped': len(unique_mapped)
        }
        
    except Exception as e:
        print(f"\n❌ 处理失败: {str(e)}")
        return {
            'success': False,
            'subject': subject_path.parent.name,
            'error': str(e)
        }

# ## 5. 批量处理所有受试者

def process_all_subjects_labels(dataset_root, save_onehot=False):
    """
    批量处理所有受试者的标签映射
    
    Parameters:
    -----------
    dataset_root : str or Path
        数据集根目录
    save_onehot : bool
        是否保存one-hot编码
    """
    dataset_root = Path(dataset_root)
    
    # 查找所有受试者的flattened目录
    subject_dirs = []
    for subject_folder in dataset_root.glob("FOR_*"):
        flattened_dir = subject_folder / 'flattened'
        if flattened_dir.exists() and (flattened_dir / 'labels.npy').exists():
            subject_dirs.append(flattened_dir)
    
    print(f"🔍 找到 {len(subject_dirs)} 个受试者的展平数据")
    
    # 处理每个受试者
    results = []
    start_time = time.time()
    
    for i, subject_dir in enumerate(subject_dirs):
        print(f"\n[{i+1}/{len(subject_dirs)}]", end='')
        result = process_subject_labels(subject_dir, save_onehot=save_onehot)
        results.append(result)
    
    # 汇总结果
    total_time = time.time() - start_time
    successful = sum(1 for r in results if r['success'])
    
    print(f"\n\n{'='*80}")
    print("📊 批量处理完成")
    print(f"{'='*80}")
    print(f"成功: {successful}/{len(subject_dirs)}")
    print(f"总耗时: {total_time:.2f}秒")
    
    # 保存处理汇总
    summary = {
        'processing_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'total_subjects': len(subject_dirs),
        'successful': successful,
        'save_onehot': save_onehot,
        'label_mapping': {
            'num_classes': 52,
            'original_labels': STANDARD_LABELS,
            'continuous_labels': list(range(52))
        },
        'results': results
    }
    
    summary_path = dataset_root / 'label_mapping_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n💾 处理汇总已保存到: {summary_path}")
    
    return results

# ## 6. 工具函数

def print_label_mapping_table():
    """
    打印完整的标签映射表
    """
    print("\n" + "="*80)
    print("完整标签映射表")
    print("="*80)
    print(f"{'连续值':>6} | {'原始值':>6}")
    print("-"*20)
    
    for continuous in range(52):
        original = REVERSE_MAPPING[continuous]
        print(f"{continuous:6d} | {original:6d}")

# ## 7. 示例使用

if __name__ == "__main__":
    # 设置数据集路径
    dataset_root = Path("/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS")
    
    # 打印映射表
    print_label_mapping_table()
    
    # 示例：处理所有受试者
    print("\n" + "="*80)
    print("开始批量处理标签映射")
    print("="*80)
    
    # 询问是否保存one-hot编码
    # 默认保存 one-hot
    save_onehot = True

    
    # 批量处理
    results = process_all_subjects_labels(dataset_root, save_onehot=save_onehot)
    
    print("\n✨ 所有标签映射处理完成！")
    
    # 使用示例
    print("\n💡 使用映射后标签的示例代码:")
    print("""
# 加载映射后的标签
import numpy as np
from pathlib import Path

subject_path = Path('...FOR_016_20250204_reproducibility/flattened')
labels_mapped = np.load(subject_path / 'labels_mapped.npy')  # 值在0-51范围
region = np.load(subject_path / 'region_mask.npy')

# 只使用非背景体素
valid_mask = region.flatten(order='F')
labels_valid = labels_mapped[valid_mask]

# 如果保存了one-hot编码
if (subject_path / 'labels_onehot_valid.npy').exists():
    labels_onehot = np.load(subject_path / 'labels_onehot_valid.npy')  # (n_valid, 52)
    print(f"One-hot shape: {labels_onehot.shape}")
    """)